# fMRI Surface Visualization (fsaverage5)

Visualizes actual fMRI data projected onto the fsaverage5 cortical surface (~20,484 vertices).

Prerequisite:
```
.venv-tribe\Scripts\python.exe scripts/project_fmri_to_surface.py --input processed_data/fMRI/Axial_MB_rsfMRI__EYES_OPEN___MSV22_.nii.gz --output-dir results/surface_projections --save-mean
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from nilearn import datasets, plotting

In [ ]:
SURFACE_DIR = Path("../results/surface_projections")
PLOT_DIR = Path("../results/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")
N_HEMI = 10242

## 1. Load Surface Data

In [ ]:
surf_data = np.load(SURFACE_DIR / "Axial_MB_rsfMRI__EYES_OPEN___MSV22__fsaverage5.npy")
print(f"Shape: {surf_data.shape}  (vertices x timepoints)")
print(f"Range: [{surf_data.min():.2f}, {surf_data.max():.2f}]")

mean_bold = surf_data.mean(axis=1)
std_bold = surf_data.std(axis=1)

## 2. Mean & Std BOLD on Cortical Surface

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), subplot_kw={"projection": "3d"})

for col, (data, label) in enumerate([(mean_bold, "Mean BOLD"), (std_bold, "Std BOLD")]):
    for row, (hemi, hl) in enumerate([("left", "Left"), ("right", "Right")]):
        plotting.plot_surf_stat_map(
            fsaverage[f"pial_{hemi}"],
            stat_map=data[:N_HEMI] if hemi == "left" else data[N_HEMI:],
            hemi=hemi, view="lateral", colorbar=True,
            title=f"{label} - {hl}", axes=axes[row, col],
        )

fig.suptitle("Resting-State fMRI on fsaverage5", fontsize=16, y=1.02)
fig.savefig(PLOT_DIR / "actual_fmri_surface.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. All Views (Lateral + Medial)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), subplot_kw={"projection": "3d"})

for col, view in enumerate(["lateral", "medial"]):
    for row, (hemi, hl) in enumerate([("left", "Left"), ("right", "Right")]):
        plotting.plot_surf_stat_map(
            fsaverage[f"pial_{hemi}"],
            stat_map=mean_bold[:N_HEMI] if hemi == "left" else mean_bold[N_HEMI:],
            hemi=hemi, view=view, colorbar=True,
            title=f"{hl} - {view}", axes=axes[row, col],
        )

fig.suptitle("Mean BOLD - All Views", fontsize=16, y=1.02)
fig.savefig(PLOT_DIR / "actual_fmri_all_views.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Temporal Dynamics

In [ ]:
lh_mean = surf_data[:N_HEMI].mean(axis=0)
rh_mean = surf_data[N_HEMI:].mean(axis=0)
time = np.arange(surf_data.shape[1])

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(time, surf_data.mean(axis=0), color="#2c3e50", linewidth=0.8)
axes[0].set_ylabel("Global mean")
axes[0].set_title("Mean BOLD across cortical surface over time")

axes[1].plot(time, lh_mean, color="#e74c3c", linewidth=0.8, label="Left")
axes[1].plot(time, rh_mean, color="#3498db", linewidth=0.8, label="Right")
axes[1].set_ylabel("Hemisphere mean")
axes[1].legend()

axes[2].plot(time, lh_mean - rh_mean, color="#27ae60", linewidth=0.8)
axes[2].axhline(0, color="gray", linewidth=0.5, linestyle="--")
axes[2].set_ylabel("L \u2212 R")
axes[2].set_xlabel("TR")

fig.tight_layout()
fig.savefig(PLOT_DIR / "temporal_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Timepoint Snapshots

In [ ]:
n_trs = surf_data.shape[1]
snapshot_trs = [0, n_trs // 4, n_trs // 2, 3 * n_trs // 4, n_trs - 1]

fig, axes = plt.subplots(len(snapshot_trs), 2, figsize=(14, 4 * len(snapshot_trs)),
                         subplot_kw={"projection": "3d"})

for i, tr in enumerate(snapshot_trs):
    for j, (hemi, hl) in enumerate([("left", "Left"), ("right", "Right")]):
        plotting.plot_surf_stat_map(
            fsaverage[f"pial_{hemi}"],
            stat_map=surf_data[:N_HEMI, tr] if hemi == "left" else surf_data[N_HEMI:, tr],
            hemi=hemi, view="lateral", colorbar=True,
            title=f"TR {tr} - {hl}", axes=axes[i, j],
        )

fig.suptitle("BOLD at Selected Timepoints", fontsize=16, y=1.01)
fig.savefig(PLOT_DIR / "timepoint_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Vertex Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mean_bold, bins=80, color="#2c3e50", alpha=0.8, edgecolor="none")
axes[0].set_title("Mean BOLD per vertex")
axes[0].set_xlabel("Mean signal")
axes[0].set_ylabel("Vertices")

axes[1].hist(std_bold, bins=80, color="#e67e22", alpha=0.8, edgecolor="none")
axes[1].set_title("BOLD Std Dev per vertex")
axes[1].set_xlabel("Std dev")

fig.tight_layout()
fig.savefig(PLOT_DIR / "vertex_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Parcellated View (Destrieux Atlas)

In [ ]:
atlas = datasets.fetch_atlas_surf_destrieux()
labels_full = np.concatenate([
    np.asarray(atlas["map_left"])[:N_HEMI],
    np.asarray(atlas["map_right"])[:N_HEMI],
])
label_names = [n.decode() if isinstance(n, bytes) else n for n in atlas["labels"]]

roi_means, roi_names = [], []
for idx, name in enumerate(label_names):
    mask = labels_full == idx
    if mask.sum() == 0 or name == "Unknown":
        continue
    roi_means.append(mean_bold[mask].mean())
    roi_names.append(name)

order = np.argsort(roi_means)
fig, ax = plt.subplots(figsize=(10, max(8, len(roi_names) * 0.22)))
ax.barh(np.arange(len(roi_means)), np.array(roi_means)[order], color="#3498db", edgecolor="none")
ax.set_yticks(np.arange(len(roi_means)))
ax.set_yticklabels(np.array(roi_names)[order], fontsize=7)
ax.set_xlabel("Mean BOLD")
ax.set_title("Mean BOLD by Destrieux Region")
fig.tight_layout()
fig.savefig(PLOT_DIR / "roi_mean_bold.png", dpi=150, bbox_inches="tight")
plt.show()